<a href="https://colab.research.google.com/github/TejashwiniVadeghar/My-Projects/blob/main/Pp1R2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os
import requests
import shutil
import tensorflow as tf
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from datetime import datetime
tf.__version__

'2.3.0'

In [ ]:
# Get features types dict and features list
content = open('/content/drive/My Drive/db/kddcup.names', 'r').readlines()
content


['back,buffer_overflow,ftp_write,guess_passwd,imap,ipsweep,land,loadmodule,multihop,neptune,nmap,normal,perl,phf,pod,portsweep,rootkit,satan,smurf,spy,teardrop,warezclient,warezmaster.\n',
 'duration: continuous.\n',
 'protocol_type: symbolic.\n',
 'service: symbolic.\n',
 'flag: symbolic.\n',
 'src_bytes: continuous.\n',
 'dst_bytes: continuous.\n',
 'land: symbolic.\n',
 'wrong_fragment: continuous.\n',
 'urgent: continuous.\n',
 'hot: continuous.\n',
 'num_failed_logins: continuous.\n',
 'logged_in: symbolic.\n',
 'num_compromised: continuous.\n',
 'root_shell: continuous.\n',
 'su_attempted: continuous.\n',
 'num_root: continuous.\n',
 'num_file_creations: continuous.\n',
 'num_shells: continuous.\n',
 'num_access_files: continuous.\n',
 'num_outbound_cmds: continuous.\n',
 'is_host_login: symbolic.\n',
 'is_guest_login: symbolic.\n',
 'count: continuous.\n',
 'srv_count: continuous.\n',
 'serror_rate: continuous.\n',
 'srv_serror_rate: continuous.\n',
 'rerror_rate: continuous.\n'

In [ ]:
buf, *features = content
attack_types = buf.split(',')
attack_types[-1] = attack_types[-1][:-1]
features_types_dict = {f.split(':')[0]: f.split(':')[1][1:-2] for f in features}
features = list(features_types_dict.keys())
features_types_dict

{'count': 'continuous',
 'diff_srv_rate': 'continuous',
 'dst_bytes': 'continuous',
 'dst_host_count': 'continuous',
 'dst_host_diff_srv_rate': 'continuous',
 'dst_host_rerror_rate': 'continuous',
 'dst_host_same_src_port_rate': 'continuous',
 'dst_host_same_srv_rate': 'continuous',
 'dst_host_serror_rate': 'continuous',
 'dst_host_srv_count': 'continuous',
 'dst_host_srv_diff_host_rate': 'continuous',
 'dst_host_srv_rerror_rate': 'continuous',
 'dst_host_srv_serror_rate': 'continuous',
 'duration': 'continuous',
 'flag': 'symbolic',
 'hot': 'continuous',
 'is_guest_login': 'symbolic',
 'is_host_login': 'symbolic',
 'land': 'symbolic',
 'logged_in': 'symbolic',
 'num_access_files': 'continuous',
 'num_compromised': 'continuous',
 'num_failed_logins': 'continuous',
 'num_file_creations': 'continuous',
 'num_outbound_cmds': 'continuous',
 'num_root': 'continuous',
 'num_shells': 'continuous',
 'protocol_type': 'symbolic',
 'rerror_rate': 'continuous',
 'root_shell': 'continuous',
 'same_

In [ ]:
# Get attack types dict
content = open('/content/drive/My Drive/db/training_attack_types', 'r').readlines()


content

['back dos\n',
 'buffer_overflow u2r\n',
 'ftp_write r2l\n',
 'guess_passwd r2l\n',
 'imap r2l\n',
 'ipsweep probe\n',
 'land dos\n',
 'loadmodule u2r\n',
 'multihop r2l\n',
 'neptune dos\n',
 'nmap probe\n',
 'perl u2r\n',
 'phf r2l\n',
 'pod dos\n',
 'portsweep probe\n',
 'rootkit u2r\n',
 'satan probe\n',
 'smurf dos\n',
 'spy r2l\n',
 'teardrop dos\n',
 'warezclient r2l\n',
 'warezmaster r2l\n',
 '\n']

In [ ]:
buf = content[:-1]
target_classes = {
    'normal': 0,
    'u2r': 1,
    'r2l': 2,
    'probe': 3,
    'dos': 4
}
attack_types_dict = {line.split()[0]: line.split()[1] for line in buf}
attack_types_dict['normal'] = 'normal'
attack_types_dict

{'back': 'dos',
 'buffer_overflow': 'u2r',
 'ftp_write': 'r2l',
 'guess_passwd': 'r2l',
 'imap': 'r2l',
 'ipsweep': 'probe',
 'land': 'dos',
 'loadmodule': 'u2r',
 'multihop': 'r2l',
 'neptune': 'dos',
 'nmap': 'probe',
 'normal': 'normal',
 'perl': 'u2r',
 'phf': 'r2l',
 'pod': 'dos',
 'portsweep': 'probe',
 'rootkit': 'u2r',
 'satan': 'probe',
 'smurf': 'dos',
 'spy': 'r2l',
 'teardrop': 'dos',
 'warezclient': 'r2l',
 'warezmaster': 'r2l'}

In [ ]:
data_file = '/content/drive/My Drive/db/kddcup.data_10_percent.gz'
data = pd.read_csv(data_file,
                      header=None,
                      names=features + ['label'])
data['label'] = [i[:-1] for i in data['label'].values]
data

,duration,protocol_type,service,flag,src_bytes,dst_bytes,land,wrong_fragment,urgent,hot,num_failed_logins,logged_in,num_compromised,root_shell,su_attempted,num_root,num_file_creations,num_shells,num_access_files,num_outbound_cmds,is_host_login,is_guest_login,count,srv_count,serror_rate,srv_serror_rate,rerror_rate,srv_rerror_rate,same_srv_rate,diff_srv_rate,srv_diff_host_rate,dst_host_count,dst_host_srv_count,dst_host_same_srv_rate,dst_host_diff_srv_rate,dst_host_same_src_port_rate,dst_host_srv_diff_host_rate,dst_host_serror_rate,dst_host_srv_serror_rate,dst_host_rerror_rate,dst_host_srv_rerror_rate,label
0,0,tcp,http,SF,181,5450,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,8,8,0.00,0.00,0.0,0.0,1.0,0.0,0.00,9,9,1.0,0.0,0.11,0.00,0.00,0.00,0.0,0.0,normal
1,0,tcp,http,SF,239,486,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,8,8,0.00,0.00,0.0,0.0,1.0,0.0,0.00,19,19,1.0,0.0,0.05,0.00,0.00,0.00,0.0,0.0,normal
2,0,tcp,http,SF,235,1337,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,8,8,0.00,0.00,0.0,0.0,1.0,0.0,0.00,29,29,1.0,0.0,0.03,0.00,0.00,0.00,0.0,0.0,normal
3,0,tcp,http,SF,219,1337,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,6,0.00,0.00,0.0,0.0,1.0,0.0,0.00,39,39,1.0,0.0,0.03,0.00,0.00,0.00,0.0,0.0,normal
4,0,tcp,http,SF,217,2032,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,6,0.00,0.00,0.0,0.0,1.0,0.0,0.00,49,49,1.0,0.0,0.02,0.00,0.00,0.00,0.0,0.0,normal
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
494016,0,tcp,http,SF,310,1881,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,4,5,0.00,0.00,0.0,0.0,1.0,0.0,0.40,86,255,1.0,0.0,0.01,0.05,0.00,0.01,0.0,0.0,normal
494017,0,tcp,http,SF,282,2286,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,6,0.00,0.00,0.0,0.0,1.0,0.0,0.00,6,255,1.0,0.0,0.17,0.05,0.00,0.01,0.0,0.0,normal
494018,0,tcp,http,SF,203,1200,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,18,0.17,0.11,0.0,0.0,1.0,0.0,0.17,16,255,1.0,0.0,0.06,0.05,0.06,0.01,0.0,0.0,normal
494019,0,tcp,http,SF,291,1200,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,6,12,0.00,0.00,0.0,0.0,1.0,0.0,0.17,26,255,1.0,0.0,0.04,0.05,0.04,0.01,0.0,0.0,normal


In [ ]:
# Unnormalized numerical features
num_attrs = []
for i in [0, 4, 5, 7, 8, 9, 12, 13, 14, 15, 16, 17, 18, 22, 31, 32]:
    num_attrs.append(features[i])
num_attrs

['duration',
 'src_bytes',
 'dst_bytes',
 'wrong_fragment',
 'urgent',
 'hot',
 'num_compromised',
 'root_shell',
 'su_attempted',
 'num_root',
 'num_file_creations',
 'num_shells',
 'num_access_files',
 'count',
 'dst_host_count',
 'dst_host_srv_count']

In [ ]:
# Categorical features
cat_attrs = []
for i in [1, 2, 3]:
    cat_attrs.append(features[i])
cat_attrs

['protocol_type', 'service', 'flag']

In [ ]:
# Classification of data
label_counts_dict = data['label'].value_counts()
print('+', '-' * 7, '+', '-' * 16, '+', '-' * 8, '+', sep='')
print('|%-6s |%-15s |%-7s |' % ('Class', "Attack type", 'Count'))
print('+', '-' * 7, '+', '-' * 16, '+', '-' * 8, '+', sep='')
for (count, attack_type) in zip(label_counts_dict, label_counts_dict.keys()):
    print('|%-6s |%-15s |%-7d |' % (attack_types_dict[attack_type], attack_type, count))
    print('+', '-' * 7, '+', '-' * 16, '+', '-' * 8, '+', sep='')

+-------+----------------+--------+
|Class  |Attack type     |Count   |
+-------+----------------+--------+
|dos    |smurf           |280790  |
+-------+----------------+--------+
|dos    |neptune         |107201  |
+-------+----------------+--------+
|normal |normal          |97278   |
+-------+----------------+--------+
|dos    |back            |2203    |
+-------+----------------+--------+
|probe  |satan           |1589    |
+-------+----------------+--------+
|probe  |ipsweep         |1247    |
+-------+----------------+--------+
|probe  |portsweep       |1040    |
+-------+----------------+--------+
|r2l    |warezclient     |1020    |
+-------+----------------+--------+
|dos    |teardrop        |979     |
+-------+----------------+--------+
|dos    |pod             |264     |
+-------+----------------+--------+
|probe  |nmap            |231     |
+-------+----------------+--------+
|r2l    |guess_passwd    |53      |
+-------+----------------+--------+
|u2r    |buffer_overflow |30

In [ ]:
# Detect small and numerous classes
numerous_classes = ['smurf', 'neptune', 'normal']
small_classes = [cl for cl in attack_types if cl not in numerous_classes]
small_classes, numerous_classes

(['back',
  'buffer_overflow',
  'ftp_write',
  'guess_passwd',
  'imap',
  'ipsweep',
  'land',
  'loadmodule',
  'multihop',
  'nmap',
  'perl',
  'phf',
  'pod',
  'portsweep',
  'rootkit',
  'satan',
  'spy',
  'teardrop',
  'warezclient',
  'warezmaster.'],
 ['smurf', 'neptune', 'normal'])

In [ ]:
train_df, test_df, val_df = data[1:2], data[2:3], data[3:4]

TRAIN_NUM = 15000
TEST_NUM = 1500

for cl in numerous_classes:
    train_df = train_df.merge(data[data['label']==cl][:TRAIN_NUM], how='outer')
    test_df = test_df.merge(data[data['label']==cl][TRAIN_NUM:TRAIN_NUM+TEST_NUM], how='outer')
    val_df = val_df.merge(data[data['label']==cl][TRAIN_NUM+TEST_NUM:TRAIN_NUM+TEST_NUM+TEST_NUM], how='outer')

for cl in small_classes:
    TRAIN_NUM = round(len(data[data['label']==cl]) * 0.8)
    TEST_NUM = round(len(data[data['label']==cl]) * 0.1)
    train_df = train_df.merge(data[data['label']==cl][:TRAIN_NUM], how='outer')
    test_df = test_df.merge(data[data['label']==cl][TRAIN_NUM:TRAIN_NUM+TEST_NUM], how='outer')
    val_df = val_df.merge(data[data['label']==cl][TRAIN_NUM+TEST_NUM:TRAIN_NUM+TEST_NUM+TEST_NUM], how='outer')

print('train_df: ', len(train_df))
print('test_df:  ', len(test_df))
print('val_df:   ', len(val_df))

train_df:  51985
test_df:   5373
val_df:    5371


In [ ]:
from sklearn.preprocessing import LabelBinarizer, MinMaxScaler

# Define a data pipeline for a RandomForestClassifier
def pipeline(data):
    df = data.copy()
    cat_encoder = LabelBinarizer()
    scaler = MinMaxScaler()
    for attr in cat_attrs:
        df[attr] = cat_encoder.fit_transform(df[attr].values.reshape(-1, 1))
    for attr in num_attrs:
        df[attr] = scaler.fit_transform(df[attr].values.reshape(-1, 1))
    return df

In [ ]:
from sklearn.ensemble import RandomForestClassifier

train_data = pipeline(train_df)
rnd_clf = RandomForestClassifier(n_estimators=500, n_jobs=-1)
rnd_clf.fit(train_data.drop('label', axis=1), train_data['label'])

RandomForestClassifier(bootstrap=True, ccp_alpha=0.0, class_weight=None,
                       criterion='gini', max_depth=None, max_features='auto',
                       max_leaf_nodes=None, max_samples=None,
                       min_impurity_decrease=0.0, min_impurity_split=None,
                       min_samples_leaf=1, min_samples_split=2,
                       min_weight_fraction_leaf=0.0, n_estimators=500,
                       n_jobs=-1, oob_score=False, random_state=None, verbose=0,
                       warm_start=False)

In [ ]:
from sklearn.metrics import accuracy_score

test_data = pipeline(test_df)
target = test_data.pop('label')
test_predicted = rnd_clf.predict(test_data)
accuracy_score(target, test_predicted)

0.9953471058998697

In [ ]:
print('+', '-' * 28, '+', '-' * 16, '+', sep='')
print('|%-27s |%-s|' % ('Feature', 'Importance(in %)'))
print('+', '-' * 28, '+', '-' * 16, '+', sep='')
for (feature, importance) in zip(features_types_dict, rnd_clf.feature_importances_):
    print('|%-28s|%-16f|' % (feature, importance * 100))
    print('+', '-' * 28, '+', '-' * 16, '+', sep='')

+----------------------------+----------------+
|Feature                     |Importance(in %)|
+----------------------------+----------------+
|duration                    |0.140977        |
+----------------------------+----------------+
|protocol_type               |7.094322        |
+----------------------------+----------------+
|service                     |0.000003        |
+----------------------------+----------------+
|flag                        |0.001706        |
+----------------------------+----------------+
|src_bytes                   |3.298295        |
+----------------------------+----------------+
|dst_bytes                   |7.178205        |
+----------------------------+----------------+
|land                        |0.013541        |
+----------------------------+----------------+
|wrong_fragment              |1.115639        |
+----------------------------+----------------+
|urgent                      |0.002162        |
+----------------------------+----------

In [ ]:
# Determine number of important features depending on their percentage of contribution
d = {i: imp for i, imp in enumerate(rnd_clf.feature_importances_)}
for bias in [0.00, 0.01, 0.02, 0.03, 0.04, 0.05, 0.06]:
    print('>', bias * 100, '%: ', len(list(filter(lambda x: d[x] > bias, d))))

> 0.0 %:  39
> 1.0 %:  23
> 2.0 %:  18
> 3.0 %:  13
> 4.0 %:  10
> 5.0 %:  6
> 6.0 %:  6


In [ ]:
def df_to_dataset(input_df, main_df=data, bias = 0.05, shuffle=True, batch_size=32):
    """
    Performs data preprocessing.

    param data: pandas.Dataframe
    """
    df = input_df.copy()
    cat_encoder = LabelBinarizer()
    scaler = MinMaxScaler()
    for attr in cat_attrs:
        cat_encoder.fit(main_df[attr].values.reshape(-1, 1))
        df[attr] = cat_encoder.transform(df[attr].values.reshape(-1, 1))
    for attr in num_attrs:
        scaler.fit(main_df[attr].values.reshape(-1, 1))
        df[attr] = scaler.fit_transform(df[attr].values.reshape(-1, 1))

    d = dict(zip(features, rnd_clf.feature_importances_)) # dict(feature: feature_importance)
    df = df.drop(list(filter(lambda x: d[x] < bias, d)), axis=1) # drop unimportance features

    df['label'] = df['label'].apply(lambda x: target_classes[attack_types_dict[x]]) # 10 classes > 4 main classes
    df['label'], _ = df['label'].factorize()

    labels = df.pop('label')
    dataset = tf.data.Dataset.from_tensor_slices((df.values, labels))
    if shuffle:
        dataset = dataset.shuffle(buffer_size=len(df))
    dataset = dataset.batch(batch_size)
    print(f'Features count: {len(df.columns)}')
    print('Сlass distribution: ', dict(labels.value_counts()))
    return dataset

In [ ]:
# Form training, test and validation datasets
BATCH_SIZE = 32
BIAS = 0.03

train_dataset = df_to_dataset(train_df, bias=BIAS, batch_size=BATCH_SIZE)
val_dataset = df_to_dataset(val_df, bias=BIAS, batch_size=BATCH_SIZE)
test_dataset = df_to_dataset(test_df,bias=BIAS, batch_size=BATCH_SIZE)

Features count: 13
Сlass distribution:  {1: 32773, 0: 15000, 4: 3286, 3: 885, 2: 41}
Features count: 13
Сlass distribution:  {1: 3346, 0: 1501, 4: 410, 3: 109, 2: 5}
Features count: 13
Сlass distribution:  {1: 3346, 0: 1501, 4: 411, 3: 110, 2: 5}


In [ ]:
# Define model
def get_compiled_model():
    model = tf.keras.Sequential([
        tf.keras.layers.Dense(14, activation='relu'),
        tf.keras.layers.Dropout(0.5),
        tf.keras.layers.Dense(10, activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(5, activation='softmax')
    ])

    model.compile(optimizer='adam',
                loss='sparse_categorical_crossentropy',
                metrics=['accuracy'])
    return model

In [ ]:
import time
model = get_compiled_model()
epochs = 100
start=time.time()
history = model.fit(train_dataset,
                    validation_data=val_dataset,
                    use_multiprocessing=True,
                    epochs=epochs,
                   )
end=time.time()
print("training time=",end-start)

Epoch 1/100

If you intended to run this layer in float32, you can safely ignore this warning. If in doubt, this warning is likely only an issue if you are porting a TensorFlow 1.X model to TensorFlow 2.

To change all layers to have dtype float64 by default, call `tf.keras.backend.set_floatx('float64')`. To change just this layer, pass dtype='float64' to the layer constructor. If you are the author of this layer, you can disable autocasting by passing autocast=False to the base Layer constructor.

1625/1625 [==============================] - 2s 1ms/step - loss: 1.3542 - accuracy: 0.7622 - val_loss: 0.2729 - val_accuracy: 0.9117
Epoch 2/100
1625/1625 [==============================] - 2s 1ms/step - loss: 0.3451 - accuracy: 0.8898 - val_loss: 0.2351 - val_accuracy: 0.9263
Epoch 3/100
1625/1625 [==============================] - 2s 1ms/step - loss: 0.2977 - accuracy: 0.9001 - val_loss: 0.2183 - val_accuracy: 0.9233
Epoch 4/100
1625/1625 [==============================] - 2s 1ms/step - lo

In [ ]:
# Check accuracy on test
import time
start=time.time()
test_loss, test_accuracy = model.evaluate(test_dataset)
end=time.time()
print('Accuracy on test dataset:', test_accuracy)
print("testing time=",end-start)

168/168 [==============================] - 0s 871us/step - loss: 0.2396 - accuracy: 0.9261
Accuracy on test dataset: 0.9261120557785034
testing time= 0.1683347225189209
